# DS 227 &middot; Knowledge Discovery in Data &mdash; Week 4 Lab
## Pulling Data from an API

Last week you scraped HTML. This week you ask a **web API** &mdash; a URL built for
programs &mdash; and it answers in clean JSON you turn straight into a table.

**How long:** about 45 minutes. Needs internet (Colab has it).

Work top to bottom. If a cell breaks, read the last line, fix it, carry on. The Stretch
section at the end is optional.

---
## Part 0 &middot; Ask, then check the status

Send a `GET` request and **check the status before you trust the body**. `200` means OK.
Run the cell.

In [ ]:
import requests

r = requests.get("https://jsonplaceholder.typicode.com/users", timeout=30)
print("status:", r.status_code)
print("type of body:", type(r.json()).__name__)
print("how many records:", len(r.json()))

**Answer here** (double-click to edit):

1. What does status `200` mean, and what might `404` or `500` tell you instead?
   &rarr; *your answer*

2. Why is checking `r.status_code` *before* `r.json()` a safer habit than parsing the body
   straight away?
   &rarr; *your answer*

---
## Part 1 &middot; JSON into a DataFrame

A list of JSON records drops straight into a DataFrame &mdash; then it is ordinary
pandas. Run the cell.

In [ ]:
import requests, pandas as pd

people = requests.get("https://jsonplaceholder.typicode.com/users", timeout=30).json()
df = pd.DataFrame(people)[["id", "name", "username", "email"]]
print(df.head())
print("\nshape:", df.shape)

**Answer here:**

1. Each JSON object became a row. Which keys did you keep as columns, and how would you
   include one more?
   &rarr; *your answer*

2. The API gave you the data over the network, yet the object you ended with is the same
   kind you have used all along. What type is it?
   &rarr; *your answer*

---
## Part 2 &middot; Pass parameters

Most APIs take **parameters** to narrow what you get. `requests` builds the query string
for you from a dict. Run the cell.

In [ ]:
import requests

r = requests.get("https://jsonplaceholder.typicode.com/comments",
                 params={"postId": 1}, timeout=30)
print("final URL:", r.url)
print("comments returned:", len(r.json()))

**Answer here:**

1. Look at `r.url`. What did `params={"postId": 1}` add to the address, and why is letting
   `requests` build that safer than pasting it by hand?
   &rarr; *your answer*

2. Change `postId` to `2` and rerun. Did the number of comments change? What does that tell
   you the parameter controls?
   &rarr; *your answer*

---
## Part 3 &middot; Be a good citizen

Someone else runs that server. Read the docs, pause between calls, and never hard-code a
secret key. Run the cell.

In [ ]:
import requests, time

ids = [1, 2, 3]
names = []
for i in ids:
    u = requests.get(f"https://jsonplaceholder.typicode.com/users/{i}", timeout=30).json()
    names.append(u["name"])
    time.sleep(0.3)      # a small pause between calls - be polite
print(names)

**Answer here:**

1. Why include `time.sleep(0.3)` between requests? What could happen to you (and the
   server) if you hammered it in a tight loop?
   &rarr; *your answer*

2. If this API needed a key, where should it live &mdash; and why never inside the notebook
   or a git commit?
   &rarr; *your answer*

---
## Stretch &mdash; optional

Stop here if you like; the required part is done.

### Stretch 1 &middot; Reach into nested JSON

Each user has a nested `address` with a `city`. Build a DataFrame of `name` and `city`
by reaching `person["address"]["city"]`.

In [ ]:
# your code here

### Stretch 2 &middot; Handle a bad request

Request `/users/99999` (which does not exist). Check the status code and print a
friendly message instead of assuming the body is valid.

In [ ]:
# your code here

---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to
download.

You need a **submit token** &mdash; one covers every lab for a month. Open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token), sign in and generate it,
then add it **once** to Colab's Secrets panel (the &#128273; icon, left sidebar) as
`LATARAK_TOKEN`. After that the cell reads it automatically, with no prompt. No Secrets
panel? The cell will just ask, hiding what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "ds227", 4

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/ds227/lab/4/submit"
    )

# The LIVE notebook, including edits you have not saved yet.
nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]

# A month-long token. Store it once in Colab Secrets (key LATARAK_TOKEN) and
# this reads it with no prompt; otherwise it asks and hides what you type.
try:
    from google.colab import userdata
    token = (userdata.get("LATARAK_TOKEN") or "").strip()
except Exception:
    token = ""
if not token:
    token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 4 submission page](https://portal.latarak.com/course/ds227/lab/4/submit) and upload it.

Re-submitting replaces your previous attempt; the most recent version is the one kept.